In [ ]:
import os

os.getcwd()

In [ ]:
coverage_file = "bamCoverage_BWA215.bedgraph"
ivar_file = "ivar_variant_BWA215.tabular"
gff3_file = "NC_004005.gff3"

In [ ]:
# ============================================================
# TYLCV variant-frequency + coverage + ORF map plot
# Input:
#   1. bamCoverage bedGraph file
#   2. iVar variant table
#   3. NC_004005.1 GFF3 annotation
# Output:
#   1. TYLCV_custom_plot.pdf
#   2. TYLCV_custom_plot.png
#   3. TYLCV_mixed_SNP_table.tsv
# ============================================================

import os
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle

# -----------------------------
# 1. Input files
# -----------------------------
coverage_file = "bamCoverage_BWA215.bedgraph"
ivar_file = "ivar_variant_BWA215.tabular"
gff3_file = "NC_004005.gff3"

chrom_name = "NC_004005.1"
genome_length = 2781

print("Current folder:", os.getcwd())
print("Files:", os.listdir())


# -----------------------------
# 2. Read GFF3 and make ORF table
# -----------------------------
def parse_gff_attributes(attr):
    d = {}
    for item in str(attr).split(";"):
        if "=" in item:
            key, value = item.split("=", 1)
            d[key] = value
    return d


gff = pd.read_csv(
    gff3_file,
    sep="\t",
    comment="#",
    header=None,
    names=[
        "chrom", "source", "type", "start", "end",
        "score", "strand", "phase", "attributes"
    ]
)

cds = gff[gff["type"] == "CDS"].copy()

cds["gene"] = cds["attributes"].apply(
    lambda x: parse_gff_attributes(x).get("gene", "CDS")
)

rename = {
    "v2": "V2",
    "cp": "V1_CP",
    "c3": "C3",
    "c2": "C2",
    "rep": "C1_Rep",
    "c4": "C4"
}

cds["name"] = cds["gene"].replace(rename)

# Convert GFF3 coordinates to BED-like coordinates:
# GFF3 is 1-based inclusive; plotting uses start/end as genomic coordinates.
orfs_bed = pd.DataFrame({
    "chrom": cds["chrom"],
    "start": cds["start"] - 1,
    "end": cds["end"],
    "name": cds["name"],
    "score": 0,
    "strand": cds["strand"]
})

# Add IR as two pieces because TYLCV is circular but plotted linearly.
# From your GFF3:
# V2 begins at 155, Rep ends at 2622, genome ends at 2781.
ir = pd.DataFrame([
    [chrom_name, 0, 154, "IR", 0, "."],
    [chrom_name, 2622, 2781, "IR", 0, "."]
], columns=["chrom", "start", "end", "name", "score", "strand"])

orfs_bed = pd.concat([orfs_bed, ir], ignore_index=True)
orfs_bed = orfs_bed.sort_values(["chrom", "start"])

orfs_bed.to_csv("orfs.bed", sep="\t", header=False, index=False)

print("ORF table:")
display(orfs_bed)


# -----------------------------
# 3. Read coverage bedGraph
# -----------------------------
cov = pd.read_csv(
    coverage_file,
    sep="\t",
    header=None,
    names=["chrom", "start", "end", "depth"]
)

# Use the bedGraph end coordinate as the plotted genome position.
cov["pos"] = cov["end"]

# Raw coverage is used, but the y-axis is capped visually at the 99th percentile
# so that extreme peaks do not flatten the whole plot.
coverage_cap = cov["depth"].quantile(0.99)

print("Coverage summary:")
display(cov["depth"].describe())
print("Coverage y-axis cap:", coverage_cap)


# -----------------------------
# 4. Read and filter iVar variants
# -----------------------------
ivar = pd.read_csv(ivar_file, sep="\t")

# iVar usually uses REGION as the reference/chromosome column.
if "REGION" in ivar.columns:
    ivar = ivar.rename(columns={"REGION": "chrom"})

ivar["POS"] = pd.to_numeric(ivar["POS"], errors="coerce")
ivar["ALT_FREQ"] = pd.to_numeric(ivar["ALT_FREQ"], errors="coerce")
ivar["TOTAL_DP"] = pd.to_numeric(ivar["TOTAL_DP"], errors="coerce")

# Keep SNPs only.
# This removes indels such as C>-AATTCAA.
snp = ivar[
    (ivar["REF"].astype(str).str.len() == 1) &
    (ivar["ALT"].astype(str).str.len() == 1)
].copy()

# Optional: keep iVar PASS calls only, if a PASS column exists.
if "PASS" in snp.columns:
    snp = snp[
        snp["PASS"].astype(str).str.upper().isin(["TRUE", "PASS"])
    ].copy()

# Filtering for mixed-population-like nucleotide frequencies.
# You can adjust these thresholds.
min_depth = 100
min_alt_freq = 0.05
max_alt_freq = 0.90

mixed_snp = snp[
    (snp["TOTAL_DP"] >= min_depth) &
    (snp["ALT_FREQ"] >= min_alt_freq) &
    (snp["ALT_FREQ"] <= max_alt_freq)
].copy()

mixed_snp["ALT_FREQ_PERCENT"] = (mixed_snp["ALT_FREQ"] * 100).round(1)


# -----------------------------
# 5. Assign each SNP to TYLCV region / ORF
# -----------------------------
def assign_region(pos, orf_table):
    hits = []
    for _, row in orf_table.iterrows():
        # orfs_bed start is 0-based, end is 1-based.
        # POS is 1-based.
        if row["start"] < pos <= row["end"]:
            hits.append(row["name"])
    if len(hits) == 0:
        return "intergenic"
    return "/".join(hits)


mixed_snp["ORF_region"] = mixed_snp["POS"].apply(
    lambda x: assign_region(x, orfs_bed)
)

# Save SNP table for paper/supplement.
mixed_snp.to_csv(
    "TYLCV_mixed_SNP_table.tsv",
    sep="\t",
    index=False
)

print("Filtered SNP table:")
display(
    mixed_snp[
        ["POS", "REF", "ALT", "TOTAL_DP", "ALT_FREQ_PERCENT", "ORF_region"]
    ].sort_values("POS")
)


# -----------------------------
# 6. Select SNVs to label
# -----------------------------
# All SNVs will still be plotted as vertical lines.
# Only selected SNVs will receive text labels.

label_min_freq = 30        # label only SNVs >=30%
max_labels = 10            # maximum number of labels
min_distance = 90          # minimum bp distance between labelled SNVs

candidates = mixed_snp[
    mixed_snp["ALT_FREQ_PERCENT"] >= label_min_freq
].copy()

# Prioritise higher-frequency SNVs
candidates = candidates.sort_values("ALT_FREQ_PERCENT", ascending=False)

selected_rows = []

for _, row in candidates.iterrows():
    pos = row["POS"]
    
    # Avoid labelling two SNVs too close together
    if all(abs(pos - selected["POS"]) >= min_distance for selected in selected_rows):
        selected_rows.append(row)
    
    if len(selected_rows) >= max_labels:
        break

label_snp = pd.DataFrame(selected_rows)

# Sort by genome position for plotting
if len(label_snp) > 0:
    label_snp = label_snp.sort_values("POS")

label_snp[["POS", "ALT", "ALT_FREQ_PERCENT", "ORF_region"]]


# -----------------------------
# 7. Plot figure
# -----------------------------
import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle
from matplotlib.ticker import FuncFormatter

# -----------------------------
# Figure style
# -----------------------------
plt.rcParams.update({
    "font.family": "Arial",     # if Arial not available, matplotlib will use fallback
    "font.size": 12,
    "axes.linewidth": 0.8,
    "pdf.fonttype": 42,
    "ps.fonttype": 42
})

fig, axes = plt.subplots(
    3, 1,
    figsize=(15, 8),
    sharex=True,
    gridspec_kw={"height_ratios": [2.2, 2.1, 1.5]}
)

ax_var, ax_cov, ax_orf = axes

# -----------------------------
# Panel 1: Alternative nucleotide frequency / SNVs
# -----------------------------
ax_var.vlines(
    mixed_snp["POS"],
    0,
    mixed_snp["ALT_FREQ_PERCENT"],
    color="black",
    linewidth=0.8
)

ax_var.set_ylim(0, 100)
ax_var.set_ylabel("Alternative nucleotide\nfrequency (%)")
ax_var.set_title(
    "Distribution of within-sample TYLCV SNVs",
    fontsize=18,
    pad=12
)

# Label selected SNVs only
# Use only percentage, or use ALT base + percentage
for i, (_, row) in enumerate(label_snp.iterrows()):
    
    # Option 1: show base and frequency
    label = f'{row["ALT"]}: {row["ALT_FREQ_PERCENT"]:.1f}%'

    # Option 2: show only frequency
    # label = f'{row["ALT_FREQ_PERCENT"]:.1f}%'
    
    # Stagger label height
    y_offset = 4 if i % 2 == 0 else 9
    
    ax_var.text(
        row["POS"],
        row["ALT_FREQ_PERCENT"] + y_offset,
        label,
        ha="center",
        va="bottom",
        fontsize=11
    )

ax_var.spines["top"].set_visible(False)
ax_var.spines["right"].set_visible(False)
ax_var.tick_params(axis="x", length=0)

# -----------------------------
# Panel 2: Read coverage
# -----------------------------
# Use raw coverage, but cap y-axis for visual clarity
coverage_cap = cov["depth"].quantile(0.99)

ax_cov.fill_between(
    cov["pos"],
    cov["depth"],
    color="#d96b6b",
    alpha=0.85,
    linewidth=0
)

ax_cov.set_ylim(0, coverage_cap)
ax_cov.set_ylabel("Read depth")

# Make y-axis easier to read, e.g. 1.0M instead of 1e6
ax_cov.yaxis.set_major_formatter(
    FuncFormatter(lambda x, pos: f"{x/1e6:.1f}M")
)

ax_cov.spines["top"].set_visible(False)
ax_cov.spines["right"].set_visible(False)
ax_cov.tick_params(axis="x", length=0)

# -----------------------------
# Panel 3: TYLCV ORF map
# -----------------------------
orf_colors = {
    "IR": "#d9d9d9",
    "V2": "#8ccf8a",
    "V1_CP": "#4daf4a",
    "C3": "#9ecae1",
    "C2": "#4292c6",
    "C1_Rep": "#9e5aa7",
    "C4": "#f28e2b"
}

for _, row in orfs_bed.iterrows():
    color = orf_colors.get(row["name"], "#bdbdbd")
    
    # Positive strand above, negative strand below, IR middle
    if row["strand"] == "+":
        y = 0.58
    elif row["strand"] == "-":
        y = 0.18
    else:
        y = 0.38
    
    height = 0.22
    
    rect = Rectangle(
        (row["start"] + 1, y),
        row["end"] - row["start"],
        height,
        facecolor=color,
        edgecolor="black",
        linewidth=0.6
    )
    ax_orf.add_patch(rect)
    
    ax_orf.text(
        (row["start"] + row["end"]) / 2,
        y + height + 0.035,
        row["name"],
        ha="center",
        va="bottom",
        fontsize=11
    )

ax_orf.set_ylim(0, 1)
ax_orf.set_yticks([])
ax_orf.set_xlabel("Genome position on NC_004005.1")
ax_orf.spines["top"].set_visible(False)
ax_orf.spines["right"].set_visible(False)
ax_orf.spines["left"].set_visible(False)

ax_orf.set_xlim(1, genome_length)

plt.tight_layout(h_pad=1.2)

plt.savefig("TYLCV_custom_plot_pretty.pdf", dpi=600, bbox_inches="tight")
plt.savefig("TYLCV_custom_plot_pretty.png", dpi=600, bbox_inches="tight")

plt.show()

In [ ]:
%pip install adjustText

In [ ]:
# ============================================================
# TYLCV mpileup nucleotide-composition figure
# Input:
#   1. BWA215.pileup
#   2. bamCoverage_BWA215.bedgraph
#   3. NC_004005.gff3
#
# Output:
#   1. TYLCV_mpileup_autoLabels_42percent.pdf
#   2. TYLCV_mpileup_autoLabels_42percent.png
#   3. TYLCV_mpileup_base_frequencies.tsv
# ============================================================

import os
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle
from matplotlib.ticker import FuncFormatter
from IPython.display import display
from adjustText import adjust_text

# -----------------------------
# Input files
# -----------------------------
mpileup_file = "BWA215.pileup"
coverage_file = "bamCoverage_BWA215.bedgraph"
gff3_file = "NC_004005.gff3"

chrom_name = "NC_004005.1"
genome_length = 2781

print("Current folder:", os.getcwd())
print("Files:", os.listdir())


# -----------------------------
# Read GFF3 and make ORF table
# -----------------------------
def parse_gff_attributes(attr):
    d = {}
    for item in str(attr).split(";"):
        if "=" in item:
            key, value = item.split("=", 1)
            d[key] = value
    return d


gff = pd.read_csv(
    gff3_file,
    sep="\t",
    comment="#",
    header=None,
    names=[
        "chrom", "source", "type", "start", "end",
        "score", "strand", "phase", "attributes"
    ]
)

cds = gff[gff["type"] == "CDS"].copy()

cds["gene"] = cds["attributes"].apply(
    lambda x: parse_gff_attributes(x).get("gene", "CDS")
)

rename = {
    "v2": "V2",
    "cp": "V1",
    "c3": "C3",
    "c2": "C2",
    "rep": "C1",
    "c4": "C4"
}

cds["name"] = cds["gene"].replace(rename)

orfs_bed = pd.DataFrame({
    "chrom": cds["chrom"],
    "start": cds["start"] - 1,
    "end": cds["end"],
    "name": cds["name"],
    "score": 0,
    "strand": cds["strand"]
})

# Add IR as two pieces because TYLCV is circular but plotted linearly
ir = pd.DataFrame([
    [chrom_name, 0, 154, "IR", 0, "."],
    [chrom_name, 2622, 2781, "IR", 0, "."]
], columns=["chrom", "start", "end", "name", "score", "strand"])

orfs_bed = pd.concat([orfs_bed, ir], ignore_index=True)
orfs_bed = orfs_bed.sort_values(["chrom", "start"])

print("ORF table:")
display(orfs_bed)


# -----------------------------
# Read coverage bedGraph
# -----------------------------
cov = pd.read_csv(
    coverage_file,
    sep="\t",
    header=None,
    names=["chrom", "start", "end", "depth"]
)

cov["pos"] = cov["end"]

# Visual cap only; raw coverage is still plotted
coverage_cap = cov["depth"].quantile(0.99)

print("Coverage summary:")
display(cov["depth"].describe())
print("Coverage y-axis cap:", coverage_cap)


# -----------------------------
# Parse mpileup read bases into A/C/G/T counts
# -----------------------------
def count_bases_from_pileup(ref_base, bases):
    """
    Convert samtools mpileup read-base string to A/C/G/T counts.

    Handles:
    . and , = reference base on forward/reverse strand
    ACGTN/acgtn = mismatches
    ^x = start of read + mapping quality character
    $ = end of read
    +numberSEQ / -numberSEQ = insertion/deletion after this base
    * and # = deletion placeholders, ignored for A/C/G/T frequency
    """
    counts = {"A": 0, "C": 0, "G": 0, "T": 0}
    ref_base = str(ref_base).upper()
    i = 0

    while i < len(bases):
        c = bases[i]

        # Start of read marker: ^ plus one mapping-quality character
        if c == "^":
            i += 2
            continue

        # End of read marker
        if c == "$":
            i += 1
            continue

        # Insertion or deletion: +12ACGT... or -3TTA
        if c in ["+", "-"]:
            i += 1
            num = ""
            while i < len(bases) and bases[i].isdigit():
                num += bases[i]
                i += 1
            if num:
                i += int(num)
            continue

        # Match to reference
        if c in [".", ","]:
            if ref_base in counts:
                counts[ref_base] += 1
            i += 1
            continue

        # Mismatch base
        if c.upper() in counts:
            counts[c.upper()] += 1
            i += 1
            continue

        # Deletion placeholders or other symbols: ignore
        i += 1

    return counts


rows = []

with open(mpileup_file) as f:
    for line in f:
        parts = line.rstrip("\n").split("\t")

        if len(parts) < 6:
            continue

        chrom = parts[0]
        pos = int(parts[1])
        ref = parts[2].upper()
        depth_reported = int(parts[3])
        bases = parts[4]

        counts = count_bases_from_pileup(ref, bases)
        total_acgt = sum(counts.values())

        if total_acgt > 0:
            freqs = {
                b: counts[b] / total_acgt * 100
                for b in ["A", "C", "G", "T"]
            }
        else:
            freqs = {b: 0 for b in ["A", "C", "G", "T"]}

        sorted_bases = sorted(freqs.items(), key=lambda x: x[1], reverse=True)
        major_base, major_freq = sorted_bases[0]
        second_base, second_freq = sorted_bases[1]

        rows.append({
            "chrom": chrom,
            "POS": pos,
            "REF": ref,
            "depth_mpileup": depth_reported,
            "A_count": counts["A"],
            "C_count": counts["C"],
            "G_count": counts["G"],
            "T_count": counts["T"],
            "ACGT_depth": total_acgt,
            "A_percent": freqs["A"],
            "C_percent": freqs["C"],
            "G_percent": freqs["G"],
            "T_percent": freqs["T"],
            "major_base": major_base,
            "major_percent": major_freq,
            "second_base": second_base,
            "second_percent": second_freq
        })

basefreq = pd.DataFrame(rows)


# -----------------------------
# Assign ORF / region
# -----------------------------
def assign_region(pos, orf_table):
    hits = []
    for _, row in orf_table.iterrows():
        if row["start"] < pos <= row["end"]:
            hits.append(row["name"])

    if len(hits) == 0:
        return "intergenic"

    return "/".join(hits)


basefreq["ORF_region"] = basefreq["POS"].apply(
    lambda x: assign_region(x, orfs_bed)
)

basefreq.to_csv(
    "TYLCV_mpileup_base_frequencies.tsv",
    sep="\t",
    index=False
)

print("Base frequency table:")
display(basefreq.head())


# -----------------------------
# Filter positions for mixed-nucleotide signal
# -----------------------------
#depth ≥ 100
#second-most nucleotide frequency ≥ 30%
min_depth= 100
min_second_freq = 30

mixed_positions = basefreq[
    (basefreq["ACGT_depth"] >= min_depth) &
    (basefreq["second_percent"] >= min_second_freq)
].copy()

print("Number of mixed positions:", len(mixed_positions))

display(
    mixed_positions[
        [
            "POS", "REF",
            "A_percent", "C_percent", "G_percent", "T_percent",
            "major_base", "major_percent",
            "second_base", "second_percent",
            "ACGT_depth", "ORF_region"
        ]
    ].sort_values("POS")
)


# -----------------------------
# Select positions to label
# -----------------------------
# All mixed positions are plotted as vertical lines.
# Only positions with minor/second nucleotide frequency >=42% are labelled.

label_threshold = 42

label_pos = mixed_positions[
    mixed_positions["second_percent"] >= label_threshold
].copy()

label_pos = label_pos.sort_values("POS")

print(f"Number of labelled positions >= {label_threshold}%:", len(label_pos))

display(
    label_pos[
        ["POS", "REF", "major_base", "major_percent",
         "second_base", "second_percent", "ACGT_depth", "ORF_region"]
    ]
)


# -----------------------------
# Plot figure
# -----------------------------
plt.rcParams.update({
    "font.family": "Arial",
    "font.size": 12,
    "axes.linewidth": 0.8,
    "pdf.fonttype": 42,
    "ps.fonttype": 42
})

fig, axes = plt.subplots(
    3, 1,
    figsize=(17, 8.4),
    sharex=True,
    gridspec_kw={"height_ratios": [2.25, 2.1, 1.45]}
)

ax_var, ax_cov, ax_orf = axes


# -----------------------------
# Panel 1: minor nucleotide frequency
# -----------------------------
ax_var.vlines(
    mixed_positions["POS"],
    0,
    mixed_positions["second_percent"],
    color="black",
    linewidth=0.8
)

# Add small dots only for labelled positions
if len(label_pos) > 0:
    ax_var.scatter(
        label_pos["POS"],
        label_pos["second_percent"],
        s=18,
        color="black",
        zorder=3
    )

ax_var.set_ylim(0, 114)
ax_var.set_ylabel("Minor nucleotide\nfrequency (%)")
ax_var.set_title(
    "Within-sample TYLCV nucleotide heterogeneity",
    fontsize=18,
    pad=12
)

# Create labels:
# Example:
# 551 G>A
# 48.4%
texts = []

for _, row in label_pos.iterrows():
    x = row["POS"]
    y = row["second_percent"]


    label = (
    f'{int(row["POS"])} {row["major_base"]}>{row["second_base"]}'
    f'\n{row["second_percent"]:.1f}%'
    )
    txt = ax_var.text(
        x,
        y + 3,
        label,
        ha="center",
        va="bottom",
        fontsize=11,
        linespacing=1.0,
        bbox=dict(
            boxstyle="round,pad=0.14",
            facecolor="white",
            edgecolor="none",
            alpha=0.78
        )
    )

    texts.append(txt)

# Automatically move labels to reduce overlap
if len(texts) > 0:
    adjust_text(
        texts,
        ax=ax_var,
        only_move={"points": "y", "text": "xy"},
        expand_text=(1.18, 1.45),
        expand_points=(1.05, 1.2),
        force_text=(0.35, 0.8),
        force_points=(0.2, 0.4),
        lim=700,
        arrowprops=dict(
            arrowstyle="-",
            color="gray",
            lw=0.55
        )
    )

ax_var.spines["top"].set_visible(False)
ax_var.spines["right"].set_visible(False)
ax_var.tick_params(axis="x", length=0)


# -----------------------------
# Panel 2: coverage
# -----------------------------
ax_cov.fill_between(
    cov["pos"],
    cov["depth"],
    color="#d97b7b",
    alpha=0.85,
    linewidth=0
)

ax_cov.set_ylim(0, coverage_cap)
ax_cov.set_ylabel("Read depth")

ax_cov.yaxis.set_major_formatter(
    FuncFormatter(lambda x, pos: f"{x/1e6:.1f}M")
)

ax_cov.spines["top"].set_visible(False)
ax_cov.spines["right"].set_visible(False)
ax_cov.tick_params(axis="x", length=0)


# -----------------------------
# Panel 3: TYLCV ORF map
# -----------------------------
orf_colors = {
    "IR": "#d9d9d9",
    "V2": "#8ccf8a",
    "V1_CP": "#4daf4a",
    "C3": "#9ecae1",
    "C2": "#4a90c2",
    "C1_Rep": "#9a5db0",
    "C4": "#f28e2b"
}

for _, row in orfs_bed.iterrows():
    color = orf_colors.get(row["name"], "#bdbdbd")

    if row["strand"] == "+":
        y = 0.58
    elif row["strand"] == "-":
        y = 0.18
    else:
        y = 0.38

    height = 0.22

    rect = Rectangle(
        (row["start"] + 1, y),
        row["end"] - row["start"],
        height,
        facecolor=color,
        edgecolor="black",
        linewidth=0.6
    )

    ax_orf.add_patch(rect)

    ax_orf.text(
        (row["start"] + row["end"]) / 2,
        y + height + 0.035,
        row["name"],
        ha="center",
        va="bottom",
        fontsize=12
    )

ax_orf.set_ylim(0, 1)
ax_orf.set_yticks([])
ax_orf.set_xlabel("Genome position (NC_004005.1)")

ax_orf.spines["top"].set_visible(False)
ax_orf.spines["right"].set_visible(False)
ax_orf.spines["left"].set_visible(False)

ax_orf.set_xlim(1, genome_length)

plt.tight_layout(h_pad=1.25)

plt.savefig(
    "TYLCV_mpileup_autoLabels_42percent.pdf",
    dpi=600,
    bbox_inches="tight"
)

plt.savefig(
    "TYLCV_mpileup_autoLabels_42percent.png",
    dpi=600,
    bbox_inches="tight"
)

plt.show()